# 7 — Generating Gaussian random fields with `PowerLawGRF`

`skewerjax.fields.toy_model_samplers.PowerLawGRF` draws real, periodic, zero-mean Gaussian random fields in 1D, 2D or 3D with the power-law spectrum

$$P(k) = A_s\,(k/k_0)^{\,n_s-1},\qquad P(0)=0 ,$$

normalised so that $\langle|\delta_k|^2\rangle = V\,P(k)$, where $\delta_k = dV\cdot\mathrm{rfftn}(\delta)$. Scalars give 1D grids; tuples of length 2 or 3 give 2D or 3D grids, which may be anisotropic. There are two ways to specify the grid:

- **(a)** `L`, `N`: box side(s) and number of cells per axis;
- **(b)** two or three of `k_min`, `k_max`, `N_k` per axis, with $k_{\min}=2\pi/L$ (fundamental), $k_{\max}=\pi N/L$ (Nyquist) and $N_k=N/2$.

The second part (**P(k) and ξ from GRFs**) shows how the package measures two-point statistics of these fields: how P(k) is computed in 1D and 3D, the recovery of the input P(k) and ξ, and the 1D–3D consistency of skewers drawn through a 3D box.

In [ ]:
import os
os.environ["JAX_PLATFORM_NAME"] = "cpu"  # must be set before importing jax

import numpy as np
import jax
jax.config.update("jax_enable_x64", True)
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

import skewerjax as sjx
from skewerjax.fields.toy_model_samplers import PowerLawGRF

font, rcnew = sjx.config.setup_matplotlib.matplotlib_default_config()
plt.rc('font', **font)
plt.rcParams.update(rcnew)

key = jax.random.PRNGKey(137)


def show_real(ax, grf, delta, axes=(0, 1), title="", fig=None):
    """Real-space panel: line (1D) or image of a 2D array with the extent of `axes`."""
    if delta.ndim == 1:
        ax.plot(grf.x_grid()[0], delta, lw=1)
        ax.set(xlabel="x [length]", ylabel=r"$\delta(x)$", title=title)
        return
    s = 3 * np.sqrt(float(grf.variance()))
    a, b = axes
    im = ax.imshow(np.asarray(delta).T, origin="lower", cmap="RdBu_r", vmin=-s, vmax=s,
                   extent=[0, grf.L[a], 0, grf.L[b]], interpolation="nearest")
    ax.set(xlabel=f"{'xyz'[a]} [length]", ylabel=f"{'xyz'[b]} [length]", title=title)
    (fig or ax.figure).colorbar(im, ax=ax, label=r"$\delta$")


def show_kspace(ax, k_a, k_b, power, labels, title=r"$|\delta_k|^2$"):
    """k-space panel: |delta_k|^2 on a log colour scale (k = 0 left blank)."""
    oa, ob = np.argsort(k_a), np.argsort(k_b)          # FFT order -> increasing k
    img = np.where(power > 0, power, np.nan)[np.ix_(oa, ob)]
    pos = img[np.isfinite(img)]
    pm = ax.pcolormesh(k_a[oa], k_b[ob], img.T, shading="nearest", cmap="Blues",
                       norm=LogNorm(vmin=np.percentile(pos, 1), vmax=pos.max()))
    ax.set(xlabel=labels[0], ylabel=labels[1], title=title)
    ax.figure.colorbar(pm, ax=ax)

In [ ]:
# 1D from (L, N)
grf1 = PowerLawGRF(A_s=1.0, k_0=0.1, n_s=0.0, L=1000.0, N=4096)
delta1, dk1 = grf1.sample(key, return_modes=True)
(k1,), _ = grf1.k_grid()
k1, p1 = np.asarray(k1)[1:], np.abs(np.asarray(dk1))[1:] ** 2
print(grf1)

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(26, 8), layout="constrained")
show_real(ax0, grf1, np.asarray(delta1), title=f"1D, N={grf1.shape[0]}")
ax1.loglog(k1, p1, ".", ms=3, alpha=0.5, label=r"$|\delta_k|^2$")
ax1.loglog(k1, grf1.volume * np.asarray(grf1.power(k1)), "k-", label=r"$V\,P(k)$")
ax1.set(xlabel="k [1/length]", ylabel=r"$|\delta_k|^2$ [length$^2$]", title="k space")
ax1.legend()
plt.show()

In [ ]:
# 2D anisotropic box from (k_min, k_max, N_k): N = 2 N_k, L = 2 pi / k_min
grf2 = PowerLawGRF(A_s=1.0, k_0=0.1, n_s=-0.5, k_min=(0.02, 0.04), k_max=(5.12, 2.56))
delta2, dk2 = grf2.sample(key, return_modes=True)
(kx2, ky2), _ = grf2.k_grid()
print(grf2)

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(30, 9), layout="constrained")
show_real(ax0, grf2, delta2, title=f"2D, N={grf2.shape}")
show_kspace(ax1, np.asarray(kx2).ravel(), np.asarray(ky2).ravel(), np.abs(np.asarray(dk2)) ** 2,
            (r"$k_x$ [1/length]", r"$k_y \geq 0$ [1/length]"))
plt.show()

In [ ]:
# 3D anisotropic box from tuple (L, N): central slice per plane + |delta_k|^2 in the k_z = 0 plane
grf3 = PowerLawGRF(A_s=1.0, k_0=0.1, n_s=-0.5, L=(100.0, 80.0, 60.0), N=(128, 96, 72))
delta3, dk3 = grf3.sample(key, return_modes=True)
d3 = np.asarray(delta3)
(kx3, ky3, _), _ = grf3.k_grid()
Nx, Ny, Nz = grf3.shape
print(grf3)

fig, axs = plt.subplots(2, 2, figsize=(26, 20), layout="constrained")
show_real(axs[0, 0], grf3, d3[:, :, Nz // 2], axes=(0, 1), title=r"slice $z=L_z/2$")
show_real(axs[0, 1], grf3, d3[:, Ny // 2, :], axes=(0, 2), title=r"slice $y=L_y/2$")
show_real(axs[1, 0], grf3, d3[Nx // 2, :, :], axes=(1, 2), title=r"slice $x=L_x/2$")
show_kspace(axs[1, 1], np.asarray(kx3).ravel(), np.asarray(ky3).ravel(),
            np.abs(np.asarray(dk3)[:, :, 0]) ** 2, (r"$k_x$ [1/length]", r"$k_y$ [1/length]"),
            title=r"$|\delta_k|^2$, $k_z=0$")
plt.show()

In [ ]:
# 2D: several spectral indices with the same key (same white noise, different filter)
ns_list = (-1.0, 0.0, 1.0, 2.0)
fig, axs = plt.subplots(2, 2, figsize=(28, 16), layout="constrained")
for ax, ns in zip(axs.flat, ns_list):
    g = PowerLawGRF(A_s=1.0, k_0=0.1, n_s=ns, L=(200.0, 100.0), N=(512, 256))
    show_real(ax, g, g.sample(key), title=rf"$n_s={ns}$,  $\sigma={np.sqrt(float(g.variance())):.3g}$")
plt.show()

## P(k) and ξ from GRFs: how they are computed and recovered

The statistics come from `OverfluxStatistics1D` and `OverfluxStatistics3D` (`skewerjax.summary_stats`). The figures come from `skewerjax.utils.plotting_tools`. Conventions (notes `subsec_P1D.tex`, in d dimensions):

$$\hat P(\mathbf k) = \frac{dV}{N}\,|\mathrm{rfftn}\,\delta|^2,\qquad \hat\xi(\mathbf r) = \frac1N\sum_{\mathbf x}\delta(\mathbf x)\,\delta(\mathbf x+\mathbf r) = \frac{\mathrm{irfftn}(\hat P)}{dV},$$

These are the same normalisations as `PowerLawGRF`, so ⟨P̂⟩ = P(k). The input ξ is the discrete inverse transform of the input P on the same grid. In 1D the field is a set of skewers with velocity units (km/s, k in s/km). In 3D, z is the line of sight, and the box uses one length unit on all axes (cMpc), so |k| and μ = k∥/|k| are defined.

The statistics classes take an overflux object. `as_overflux` below wraps a stack of GRF skewers as a single-line overflux (no metal terms).

In [ ]:
from types import SimpleNamespace
import jax.numpy as jnp
import skewerjax.utils.plotting_tools as pt
from skewerjax.summary_stats.P1D_xi1D import OverfluxStatistics1D
from skewerjax.summary_stats.P3D_xi3D import OverfluxStatistics3D
from skewerjax.summary_stats.two_point import xi_from_power, xi3d_from_power


def as_overflux(skewers):
    # (N_skewers, N_pix) field -> the attributes the statistics classes read
    d = jnp.asarray(skewers)
    return SimpleNamespace(subset_keys=["a"], delta_terms={"a": d}, C_per_sample=jnp.zeros((d.shape[0], 1)), total=d)


N_SK, N_REAL = 256, 8                      # 1D skewers; 3D realisations

### 1D: from skewers to P1D
- **Left:** each skewer δ(v) gives a periodogram P_s(k) = (dv/N_v)|rfft δ_s|². A single mode scatters as P χ²₂/2 around the input, and the mean over skewers converges to it.
- **Right:**
  - The k bins are linear at low k, one mode per bin at (j+½)k_f (`hybrid_k_edges`), and logarithmic above.
  - `bin_p1d` returns the mode-weighted mean and the number of modes, 2 per skewer per k (±k) and 1 at Nyquist. That count sets the Gaussian error σ/P = √(2/n_modes).

In [ ]:
g1 = PowerLawGRF(A_s=1.0, k_0=0.05, n_s=0.5, L=2000.0, N=512)          # L [km/s]
sk1 = jax.vmap(g1.sample)(jax.random.split(jax.random.fold_in(key, 1), N_SK))
st1 = OverfluxStatistics1D(as_overflux(sk1), g1.dx[0]).compute_xi1d()
e1 = pt.hybrid_k_edges(g1.k_min[0], g1.k_max[0])

fig, axs = pt.plot_p1d_from_skewers(st1, power=g1.power)
plt.show()
fig, axs = pt.plot_p1d_mode_binning(st1, e1, power=g1.power)
plt.show()

### 1D recovery of P and ξ
- **P1D:** the binned P1D is compared with the input averaged over the same bins, with Gaussian errors from `n_modes`.
- **ξ1D:** `xi_total` (the mean over skewers) is compared with ξ_in = irfft(P_in)/dv, with errors from the scatter over skewers.
  - The lower panel shows pulls, because ξ crosses zero.
  - Neighbouring lags are correlated, so χ²/dof is only indicative for ξ.

In [ ]:
P1, kb1, _, nb1 = st1.bin_p1d(e1)
P1_in = st1.bin_p1d(e1, Pk=g1.power(st1.k))[0]
xi1_in = np.fft.fftshift(np.asarray(xi_from_power(g1.power(st1.k), g1.shape[0], g1.dx[0])))
_, xi1_err = pt.sample_mean_and_error(st1.xi_total_per_sample)

fig, axs = pt.plot_pk_recovery((P1, kb1, e1, nb1), P1_in, power=g1.power, label=f"{N_SK} skewers")
plt.show()
fig, axs = pt.plot_xi_recovery(st1, xi1_in, sigma=xi1_err, max_lag=300.0)
plt.show()
print(pt.recovery_summary(P1, P1_in, pt.pk_sigma(P1_in, nb1), "1D P(k)"))
print(pt.recovery_summary(st1.xi_total, xi1_in, xi1_err, "1D xi   ", ratio=False))

### 3D: k grid and (k⊥, k∥) binning
The figures below use an anisotropic box with L = (90, 60, 120) and N = (36, 30, 48), and `N_REAL` realisations. Each realisation is read as Nx·Ny skewers along z.
1. Central slices of one realisation.
2. P̂ of single modes on the k∥ = 0, k_y = 0 and k_x = 0 planes. The spectra use the rfft layout, so k∥ ≥ 0. The mode lattice has spacing 2π/L_i per axis, so it is rectangular in this box.
3. All modes in 3D, with k⊥ cylinders and k∥ planes.
4. The (k⊥, k∥) cells of `bin_p3d_kperp` and the number of modes in each cell.
5. P(k⊥, k∥), the mean over realisations. Every cell is then plotted against |k| = √(k⊥²+k∥²): for this isotropic input they collapse onto P(|k|) whatever their μ².

In [9]:
g3 = PowerLawGRF(A_s=1.0, k_0=0.05, n_s=0.5, L=(90.0, 60.0, 120.0), N=(36, 30, 48))
(Nx, Ny, Nz), (dx, dy, dz) = g3.shape, g3.dx
U = pt.UNITS_3D_COMOVING
f3 = jax.vmap(g3.sample)(jax.random.split(jax.random.fold_in(key, 3), N_REAL))
skewers3 = [f.reshape(Nx * Ny, Nz) for f in f3]                          # skewers along z
st3 = [OverfluxStatistics3D(as_overflux(s), dz, (dx, dy), grid_shape=(Nx, Ny)).compute_xi3d() for s in skewers3]
e3 = pt.hybrid_k_edges(min(g3.k_min[:2]), min(g3.k_max[:2]), n_lin=6, n_log=8)

fig, axs = pt.plot_field_slices(f3[0], g3, units=U)
plt.show()
fig, axs = pt.plot_kspace_slices(st3[0], units=U)
plt.show()
fig, ax = pt.plot_kspace_3d(st3[0], k_perp_radii=e3[[3, 7, -1]], units=U)
plt.show()
fig, axs = pt.plot_kperp_kpar_binning(st3[0], e3, units=U)
plt.show()

bins3 = [s.bin_p3d_kperp(k_perp_edges=e3) for s in st3]
binned3 = (pt.sample_mean_and_error([b[0] for b in bins3])[0],) + tuple(bins3[0][1:])
fig, ax = pt.plot_p3d_kperp_kpar(binned3, units=U)
plt.show()
fig, ax = pt.plot_p3d_cells(binned3, x="kmod", reference=g3.power, n_real=N_REAL, units=U)
plt.show()

### 3D recovery of P and ξ
- **P3D:** the spherically binned P3D (`bin_p3d_kmod`, mean over realisations) is compared with the bin-averaged input, with Gaussian errors from the mode counts.
- **ξ3D:** the realisation-mean ξ3D is compared with irfftn(P_in)/dV along the three axes through r = 0, with errors from the scatter over realisations. The summary line uses every lag cell. With only `N_REAL` = 8 realisations the error estimate is itself noisy, so the pulls follow a Student-t distribution with 7 degrees of freedom: about 91% fall within 2σ, and the largest of the ~5×10⁴ pulls can reach about 9.

In [10]:
ked3 = np.geomspace(0.999 * min(g3.k_min), min(g3.k_max), 13)
P3, _ = pt.sample_mean_and_error([s.bin_p3d_kmod(ked3)[0] for s in st3])
_, kb3, _, nb3 = st3[0].bin_p3d_kmod(ked3)
_, kmod3 = g3.k_grid(real=True)
P3d_in = g3.power(kmod3)                                              # input on the rfft grid
P3_in = st3[0].bin_p3d_kmod(ked3, Pk3d_input=P3d_in)[0]
xi3, xi3_err = pt.sample_mean_and_error([s.xi3d_total for s in st3])
xi3_in = np.asarray(xi3d_from_power(P3d_in, g3.shape, g3.cell_volume))

fig, axs = pt.plot_p3d_recovery((P3, kb3, ked3, nb3), P3_in, power=g3.power, n_real=N_REAL, units=U,
                                label=f"{N_REAL} realisations")
plt.show()
fig, axs = pt.plot_xi3d_recovery(st3[0], xi3_in, xi=xi3, sigma=xi3_err, max_lag=40.0, units=U)
plt.show()
print(pt.recovery_summary(P3, P3_in, pt.pk_sigma(P3_in, nb3, N_REAL), "3D P(|k|)"))
print(pt.recovery_summary(xi3, xi3_in, xi3_err, "3D xi     ", ratio=False))

### 1D–3D consistency
The same Nx·Ny skewers along z, measured with `OverfluxStatistics1D`, must satisfy:
- **Exactly, per realisation (Parseval over the transverse plane):** P1D(k_z) = Σ_{k⊥} P̂3D(k⊥, k_z)/(L_x L_y), and ξ1D(Δ) = ξ3D(r⊥ = 0, Δ).
- **Statistically:** the mean P1D follows the projected input P1D_in(k_z) = Σ_{k⊥} P_in(√(k⊥²+k_z²))/(L_x L_y).

Skewers in one box are correlated, so the errors come from the scatter over realisations, not from the mode counts.

In [11]:
A_perp = g3.L[0] * g3.L[1]
st13 = [OverfluxStatistics1D(as_overflux(s), dz).compute_xi1d() for s in skewers3]
rel = lambda a, b: float(np.max(np.abs(np.asarray(a) - np.asarray(b))) / np.max(np.abs(np.asarray(b))))
err_P = max(rel(s1.Pk_total, np.asarray(s3.Pk3d_total).sum(axis=(0, 1)) / A_perp) for s1, s3 in zip(st13, st3))
err_xi = max(rel(s1.xi_total, np.asarray(s3.xi3d_total)[Nx // 2, Ny // 2]) for s1, s3 in zip(st13, st3))

P1_proj_in = np.asarray(P3d_in).sum(axis=(0, 1)) / A_perp                  # projected input P1D(k_z)
ez = pt.hybrid_k_edges(g3.k_min[2], g3.k_max[2], n_lin=6, n_log=8)
P13, P13_err = pt.sample_mean_and_error([s.bin_p1d(ez)[0] for s in st13])
_, kbz, _, nbz = st13[0].bin_p1d(ez)
P13_in = st13[0].bin_p1d(ez, Pk=P1_proj_in)[0]

fig, axs = pt.plot_pk_recovery((P13, kbz, ez, nbz), P13_in, sigma=P13_err, label="P1D of z-skewers",
                               k_label=r"$k_z\ [\mathrm{cMpc^{-1}}]$", P_label=r"$P_{\rm 1D}(k_z)\ [\mathrm{cMpc}]$")
axs[0].plot(kbz, P13_in, "k-", lw=2, label=r"projected input $\sum_{k_\perp} P_{\rm in}/(L_xL_y)$")
axs[0].legend(frameon=False, fontsize="small")
plt.show()
print(f"exact identities over {N_REAL} realisations: max rel |P1D - sum P3D / (Lx Ly)| = {err_P:.1e}, "
      f"max rel |xi1D - xi3D(r_perp=0)| = {err_xi:.1e}")
print(pt.recovery_summary(P13, P13_in, P13_err, "1D-3D P1D vs projected P3D"))